# KNN and K-Means from Scratch

Contrast supervised neighbor voting with unsupervised centroid updates using the same pairwise-distance primitive.

- **Study time:** 40-50 minutes
- **Prerequisites:** NumPy broadcasting, distances, and estimator state
- **Mode:** `quick`
- **Data policy:** no external files or downloads; seeded synthetic arrays only
- **Provenance:** consolidated from the legacy K-means implementation and NumPy distance patterns; KNN added to fill a curriculum gap

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, subtle API behavior, or configuration side effects; obvious Python syntax is left uncommented.


In [1]:
import numpy as np

from datacoding.algorithms import KMeans, KNNClassifier

rng = np.random.default_rng(31)  # Reproducible local generator without global RNG side effects.

## 1. The shared pairwise-distance primitive


In [2]:
queries = np.array([[0.0, 0.0], [2.0, 2.0]])
references = np.array([[0.0, 1.0], [1.0, 0.0], [3.0, 3.0]])
squared_distances = np.sum(
    (queries[:, None, :] - references[None, :, :])
    ** 2,  # Broadcast to (n_queries, n_references, d).
    axis=2,
)  # Squared distance preserves neighbor ordering without computing square roots.

print("Distances | query/reference shapes", (queries.shape, references.shape))
print("Distances | pairwise squared matrix", squared_distances)

Distances | query/reference shapes ((2, 2), (3, 2))
Distances | pairwise squared matrix [[ 1.  1. 18.]
 [ 5.  5.  2.]]


## 2. KNN stores examples and votes at prediction time


In [3]:
X_train = np.array([[0, 0], [0, 1], [1, 0], [9, 9], [9, 10], [10, 9]], dtype=float)
y_train = np.array(["low"] * 3 + ["high"] * 3)
knn = KNNClassifier(n_neighbors=3).fit(X_train, y_train)
X_query = np.array([[0.2, 0.1], [9.4, 9.2]])

print("KNN | retained training shape", knn.X_train_.shape)
print(
    "KNN | neighbor indices", knn._neighbor_indices(X_query)
)  # Private helper inspected only to expose the voting mechanism.
print("KNN | predictions", knn.predict(X_query))

KNN | retained training shape (6, 2)
KNN | neighbor indices [[0 2 1]
 [3 5 4]]
KNN | predictions ['low' 'high']


## 3. Feature scale can redefine nearest

This isolated geometry demo standardizes its full toy matrix. In a supervised workflow, learn mean and standard deviation from the training split only.


In [4]:
scale_demo = np.array([[0.0, 1.0], [1.0, 1000.0], [2.0, 1100.0]])
mean = scale_demo.mean(axis=0)
std = scale_demo.std(axis=0) + 1e-12  # Keep a constant feature from producing an infinite scale.
standardized = (scale_demo - mean) / std  # Put both features on comparable scales.

raw_distance = np.sum((scale_demo[0] - scale_demo[1:]) ** 2, axis=1)
standardized_distance = np.sum((standardized[0] - standardized[1:]) ** 2, axis=1)
print("Scaling | raw squared distances", raw_distance)
print("Scaling | standardized squared distances", standardized_distance)

Scaling | raw squared distances [ 998002. 1207805.]
Scaling | standardized squared distances [ 5.55361535 10.9057673 ]


## 4. K-means alternates assignment and update


In [5]:
X = np.vstack(  # Three compact clouds; labels are intentionally absent.
    [
        rng.normal([0, 0], 0.25, size=(100, 2)),
        rng.normal([5, 0], 0.25, size=(100, 2)),
        rng.normal([2.5, 4], 0.25, size=(100, 2)),
    ]
)
kmeans = KMeans(n_clusters=3, max_iter=100, random_state=31).fit(X)

cluster_counts = np.bincount(
    kmeans.labels_, minlength=3
)  # Keep an explicit zero slot for any empty cluster ID.
print("K-means | learned centers", kmeans.cluster_centers_)
print("K-means | cluster counts", cluster_counts)
print("K-means | inertia", kmeans.inertia_)
print("K-means | iterations", kmeans.n_iter_)

K-means | learned centers [[ 5.04646086  0.02294978]
 [ 0.00784206 -0.01229208]
 [ 2.50177518  4.0514132 ]]
K-means | cluster counts [100 100 100]
K-means | inertia 38.540920394608584
K-means | iterations 2


## 5. One update written explicitly


In [6]:
initial_centers = X[
    [0, 100, 200]
].copy()  # One seed per known toy cloud keeps this single update interpretable.
distances = np.sum(
    (X[:, None, :] - initial_centers[None, :, :]) ** 2, axis=2
)  # Assignment costs (n, k).
labels = np.argmin(distances, axis=1)  # Assign each row to its nearest center.
updated_centers = np.vstack(
    [X[labels == cluster].mean(axis=0) for cluster in range(3)]
)  # Replace every center with its assigned-row mean.

print("One K-means iteration | initial centers", initial_centers)
print("One K-means iteration | updated centers", updated_centers)

One K-means iteration | initial centers [[-0.09882532  0.06597872]
 [ 4.96485841 -0.01564076]
 [ 2.7139898   3.62085356]]
One K-means iteration | updated centers [[ 0.00784206 -0.01229208]
 [ 5.04646086  0.02294978]
 [ 2.50177518  4.0514132 ]]


## 6. Retrieval checks


In [7]:
assert knn.predict(X_query).tolist() == ["low", "high"]
assert len(np.unique(kmeans.labels_)) == 3
assert squared_distances.shape == (2, 3)
assert np.all(cluster_counts > 0)

print("Distance-algorithm checks | status", "all assertions passed")

Distance-algorithm checks | status all assertions passed
